# OpenVLA dual-condition smoke test v4

Clean rebuild of the pre-institutional smoke test.

Each condition trains for only **12 successful optimiser steps**. This validates the
experiment engine, checkpoint format and reload path, not policy quality.

Heavy temporary files live under `/kaggle/tmp`; only the small final JSON report is
written to `/kaggle/working`.

## 1. Configuration and storage preflight

In [1]:
from pathlib import Path
import os, sys, shutil, json, subprocess, ast

SCRATCH = Path("/kaggle/tmp/openvla_dual_smoke_v4")
PERSIST = Path("/kaggle/working/openvla_dual_smoke_v4_results")
OPENVLA_DIR = SCRATCH / "openvla"
DATA_DIR = SCRATCH / "datasets" / "openvla" / "modified_libero_rlds"
HF_HOME = SCRATCH / "hf_cache"
RUNS_DIR = SCRATCH / "runs"

OPENVLA_COMMIT = "c8f03f48af692657d3060c19588038c7220e9af9"
MMD_SELECTED_LAYERS = [
    9, 10, 11, 12, 13, 14, 15, 16,
    17, 18, 19, 20, 21, 22, 23, 25,
]
EXPECTED_TRAINABLE_PARAMS = 110_828_288
SMOKE_STEPS = 12
GRAD_ACCUM = 4
BATCH_SIZE = 1
SEED = 7

if SCRATCH.exists():
    shutil.rmtree(SCRATCH)

for p in [SCRATCH, PERSIST, DATA_DIR, HF_HOME, RUNS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

scratch_free = shutil.disk_usage("/kaggle/tmp").free / 1024**3
working_free = shutil.disk_usage("/kaggle/working").free / 1024**3
print(f"/kaggle/tmp free:     {scratch_free:.1f} GiB")
print(f"/kaggle/working free: {working_free:.1f} GiB")

if scratch_free < 25:
    raise RuntimeError(
        f"Only {scratch_free:.1f} GiB free in /kaggle/tmp. "
        "Need at least 25 GiB."
    )

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["TRANSFORMERS_CACHE"] = str(HF_HOME / "transformers")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_MODE"] = "disabled"

print("Storage preflight: PASS")

/kaggle/tmp free:     1119.2 GiB
/kaggle/working free: 19.5 GiB
Storage preflight: PASS


## 2. Install dependencies

In [2]:
def install(packages, no_deps=False):
    cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if no_deps:
        cmd.append("--no-deps")
    cmd.extend(packages)
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

install([
    "transformers==4.40.1",
    "tokenizers==0.19.1",
    "timm==0.9.10",
    "peft==0.11.1",
    "accelerate==0.30.1",
    "sentencepiece==0.1.99",
    "draccus==0.8.0",
    "json-numpy",
])
install([
    "einops", "jsonlines", "rich", "matplotlib",
    "huggingface_hub", "wandb",
])
install(["bitsandbytes==0.50.2"])
install(["tensorflow_datasets==4.9.3"])
install(["git+https://github.com/moojink/dlimp_openvla"], no_deps=True)

print("Dependency installation: PASS")

$ /usr/bin/python3 -m pip install -q transformers==4.40.1 tokenizers==0.19.1 timm==0.9.10 peft==0.11.1 accelerate==0.30.1 sentencepiece==0.1.99 draccus==0.8.0 json-numpy
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.7/66.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.5 MB/s eta 0:00:00
$ /usr/bin/python3 -m pip install -q e

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.1 which is incompatible.


$ /usr/bin/python3 -m pip install -q bitsandbytes==0.50.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.5 MB/s eta 0:00:00
$ /usr/bin/python3 -m pip install -q tensorflow_datasets==4.9.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 64.3 MB/s eta 0:00:00
$ /usr/bin/python3 -m pip install -q --no-deps git+https://github.com/moojink/dlimp_openvla
Dependency installation: PASS


## 3. Pin OpenVLA and patch T4 compatibility

In [3]:
import re
import py_compile
import accelerate

subprocess.run(
    ["git", "clone", "https://github.com/openvla/openvla.git", str(OPENVLA_DIR)],
    check=True,
)
subprocess.run(
    ["git", "checkout", "--detach", OPENVLA_COMMIT],
    cwd=OPENVLA_DIR,
    check=True,
)
actual = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=OPENVLA_DIR,
    text=True,
).strip()
assert actual == OPENVLA_COMMIT

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(OPENVLA_DIR), "--no-deps"],
    check=True,
)

droid_file = (
    OPENVLA_DIR / "prismatic" / "vla" / "datasets" / "rlds" /
    "oxe" / "utils" / "droid_utils.py"
)
txt = droid_file.read_text()
needle = "import tensorflow_graphics.geometry.transformation as tfg"
if needle in txt:
    droid_file.write_text(txt.replace(needle, f"# {needle}"))

accelerate_file = Path(accelerate.__file__).parent / "big_modeling.py"
src = accelerate_file.read_text()

old = '        if device != "disk":\n            model.to(device)\n'
new = (
    '        if device != "disk":\n'
    '            if not getattr(model, "is_quantized", False):\n'
    '                model.to(device)\n'
)

if new not in src:
    if old in src:
        src = src.replace(old, new, 1)
    else:
        pat = re.compile(
            r'(\n(\s*)if device != "disk":\n)(\s*)model\.to\(device\)\n'
        )
        def repl(m):
            body = m.group(3)
            return (
                f'{m.group(1)}'
                f'{body}if not getattr(model, "is_quantized", False):\n'
                f'{body}    model.to(device)\n'
            )
        src, n = pat.subn(repl, src, count=1)
        if n != 1:
            raise RuntimeError("Could not patch Accelerate dispatch_model()")
    accelerate_file.write_text(src)

py_compile.compile(str(accelerate_file), doraise=True)

print("OpenVLA commit:", actual)
print("Accelerate compile check: PASS")
print("Compatibility patches: PASS")

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Cloning into '/kaggle/tmp/openvla_dual_smoke_v4/openvla'...
HEAD is now at c8f03f4 Update README: Remove code block typo


Obtaining file:///kaggle/tmp/openvla_dual_smoke_v4/openvla
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for openvla (pyproject.toml): started
  Building editable for openvla (pyproject.toml): finished with status 'done'
  Created wheel for openvla: filename=openvla-0.0.3-0.editable-py3-none-any.whl size=17781 sha256=9f30ff37da264e94e3aefffab3f829adbfaced24c92e039728b6ac735273e676
  Stored in directory: /tmp/pip-ephem-wheel-cache-y95u9otb/wheels/d6/c6/98/0d6f8dce8675d3dd697871962c4dbf107c10ccf35163876bb3
Successfully built o

## 4. Hardware/package audit

In [4]:
import torch
import transformers
import peft
import accelerate
import bitsandbytes as bnb
import tensorflow as tf
import tensorflow_datasets as tfds
from dlimp import DLataset

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bnb.__version__)
print("TensorFlow:", tf.__version__)
print("TFDS:", tfds.__version__)

assert transformers.__version__ == "4.40.1"
assert peft.__version__ == "0.11.1"
assert accelerate.__version__ == "0.30.1"
assert bnb.__version__ == "0.50.2"
assert torch.cuda.is_available()

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}, {p.total_memory/1024**3:.2f} GiB")

assert torch.cuda.get_device_capability(0) == (7, 5)
print("Hardware/package audit: PASS")

Python: 3.12.13
PyTorch: 2.10.0+cu128 CUDA: 12.8
Transformers: 4.40.1
PEFT: 0.11.1
Accelerate: 0.30.1
bitsandbytes: 0.50.2
TensorFlow: 2.20.0
TFDS: 4.9.3
GPU 0: Tesla T4, 14.56 GiB
GPU 1: Tesla T4, 14.56 GiB
Hardware/package audit: PASS


## 5. Download LIBERO-Object RLDS

In [5]:
dataset_path = DATA_DIR / "libero_object_no_noops"

subprocess.run([
    "huggingface-cli", "download",
    "openvla/modified_libero_rlds",
    "--repo-type", "dataset",
    "--include", "libero_object_no_noops/*",
    "--local-dir", str(DATA_DIR),
], check=True)

assert dataset_path.exists(), dataset_path
print("Dataset:", dataset_path)
print("Dataset download: PASS")

⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.


Fetching 34 files:   0%|          | 0/34 [00:00<?, ?it/s]Downloading 'libero_object_no_noops/1.0.0/libero_object-train.tfrecord-00005-of-00032' to '/kaggle/tmp/openvla_dual_smoke_v4/datasets/openvla/modified_libero_rlds/.cache/huggingface/download/libero_object_no_noops/1.0.0/Tkg1yWClHj8iYuw7237ljW3n6bQ=.87a67f60af45aa7f3dbc4efcb22fba1a2f0bef21123074775ab0068a6e48ed16.incomplete'
Still waiting to acquire lock on /kaggle/tmp/openvla_dual_smoke_v4/datasets/openvla/modified_libero_rlds/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Download complete. Moving file to /kaggle/tmp/openvla_dual_smoke_v4/datasets/openvla/modified_libero_rlds/libero_object_no_noops/1.0.0/dataset_info.json
Fetching 34 files:   3%|▎         | 1/34 [00:00<00:04,  7.80it/s]Download complete. Moving file to /kaggle/tmp/openvla_dual_smoke_v4/datasets/openvla/modified_libero_rlds/libero_object_no_noops/1.0.0/features.json
Download complete. Moving file to /kaggle/tmp/openvla_dual_smoke_v4/datasets/openvla/mo

/kaggle/tmp/openvla_dual_smoke_v4/datasets/openvla/modified_libero_rlds
Dataset: /kaggle/tmp/openvla_dual_smoke_v4/datasets/openvla/modified_libero_rlds/libero_object_no_noops
Dataset download: PASS


Download complete. Moving file to /kaggle/tmp/openvla_dual_smoke_v4/datasets/openvla/modified_libero_rlds/libero_object_no_noops/1.0.0/libero_object-train.tfrecord-00031-of-00032
Fetching 34 files: 100%|██████████| 34/34 [00:14<00:00,  2.42it/s]


## 6. Write and compile the common smoke-training engine

In [6]:
TRAIN_SCRIPT = '\nimport argparse\nimport csv\nimport gc\nimport json\nimport os\nimport random\nimport re\nfrom collections import deque\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nimport tqdm\nfrom accelerate import PartialState\nfrom peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.optim import AdamW\nfrom torch.utils.data import DataLoader\nfrom transformers import (\n    AutoConfig,\n    AutoImageProcessor,\n    AutoModelForVision2Seq,\n    AutoProcessor,\n    BitsAndBytesConfig,\n)\nfrom transformers.modeling_outputs import CausalLMOutputWithPast\n\nfrom prismatic.extern.hf.configuration_prismatic import OpenVLAConfig\nfrom prismatic.extern.hf.modeling_prismatic import OpenVLAForActionPrediction\nfrom prismatic.extern.hf.processing_prismatic import (\n    PrismaticImageProcessor,\n    PrismaticProcessor,\n)\nfrom prismatic.models.backbones.llm.prompting import PurePromptBuilder\nfrom prismatic.util.data_utils import PaddedCollatorForActionPrediction\nfrom prismatic.vla.action_tokenizer import ActionTokenizer\nfrom prismatic.vla.datasets import RLDSBatchTransform, RLDSDataset\nfrom prismatic.vla.datasets.rlds.utils.data_utils import save_dataset_statistics\n\nMMD_SELECTED_LAYERS = [\n    9, 10, 11, 12, 13, 14, 15, 16,\n    17, 18, 19, 20, 21, 22, 23, 25,\n]\nEXPECTED_TRAINABLE = 110_828_288\n\n\ndef build_lora_config(condition):\n    if condition == "uniform":\n        return LoraConfig(\n            r=32,\n            lora_alpha=16,\n            lora_dropout=0.0,\n            target_modules="all-linear",\n            init_lora_weights="gaussian",\n        )\n\n    lp = "|".join(map(str, MMD_SELECTED_LAYERS))\n    llm_full = (\n        rf"language_model\\.model\\.layers\\.({lp})\\."\n        r"(self_attn\\.(q_proj|k_proj|v_proj|o_proj)|"\n        r"mlp\\.(gate_proj|up_proj|down_proj))"\n    )\n    fixed_tail = (\n        r"(vision_backbone|projector)\\..*(fc1|fc2|fc3|proj|q|kv|qkv)"\n        r"|language_model\\.lm_head"\n    )\n    target_regex = rf"^({llm_full}|{fixed_tail})$"\n    rank_key = (\n        rf"layers\\.({lp})\\."\n        r"(self_attn\\.(q_proj|k_proj|v_proj|o_proj)|"\n        r"mlp\\.(gate_proj|up_proj|down_proj))"\n    )\n    return LoraConfig(\n        r=32,\n        lora_alpha=16,\n        lora_dropout=0.0,\n        target_modules=target_regex,\n        rank_pattern={rank_key: 64},\n        alpha_pattern={rank_key: 32},\n        init_lora_weights="gaussian",\n    )\n\n\ndef audit_loaded_peft(model, condition):\n    wrapped = {}\n    layer_re = re.compile(r"language_model\\.model\\.layers\\.(\\d+)\\.")\n\n    for name, module in model.named_modules():\n        if hasattr(module, "lora_A") and "default" in module.lora_A:\n            wrapped[name] = {\n                "rank": int(module.r["default"]),\n                "alpha": int(module.lora_alpha["default"]),\n            }\n\n    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    if trainable != EXPECTED_TRAINABLE:\n        raise RuntimeError(\n            f"Trainable parameter mismatch: {trainable:,} != {EXPECTED_TRAINABLE:,}"\n        )\n\n    observed_layers = sorted({\n        int(layer_re.search(name).group(1))\n        for name in wrapped\n        if layer_re.search(name) is not None\n    })\n\n    expected_layers = list(range(32)) if condition == "uniform" else MMD_SELECTED_LAYERS\n    if observed_layers != expected_layers:\n        raise RuntimeError(\n            f"{condition}: decoder-layer coverage mismatch: {observed_layers}"\n        )\n\n    for name, info in wrapped.items():\n        lm = layer_re.search(name)\n        if lm is None:\n            expected = {"rank": 32, "alpha": 16}\n        elif condition == "uniform":\n            expected = {"rank": 32, "alpha": 16}\n        else:\n            expected = {"rank": 64, "alpha": 32}\n\n        if info != expected:\n            raise RuntimeError(\n                f"{condition}: wrong LoRA config for {name}: {info}, expected {expected}"\n            )\n\n    print(\n        f"ACTUAL PEFT STRUCTURE PASS: condition={condition}, "\n        f"trainable={trainable:,}, layers={observed_layers}",\n        flush=True,\n    )\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--condition", choices=["uniform", "mmd_selective"], required=True)\n    ap.add_argument("--data-root", type=Path, required=True)\n    ap.add_argument("--output-root", type=Path, required=True)\n    ap.add_argument("--steps", type=int, default=12)\n    ap.add_argument("--grad-accum", type=int, default=4)\n    ap.add_argument("--batch-size", type=int, default=1)\n    ap.add_argument("--lr", type=float, default=5e-4)\n    ap.add_argument("--seed", type=int, default=7)\n    args = ap.parse_args()\n\n    condition = args.condition\n    run_dir = args.output_root / condition\n    adapter_dir = run_dir / "final_adapter"\n    run_dir.mkdir(parents=True, exist_ok=True)\n\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    torch.cuda.manual_seed_all(args.seed)\n\n    distributed_state = PartialState()\n    device_id = distributed_state.local_process_index\n    torch.cuda.set_device(device_id)\n    torch.cuda.empty_cache()\n\n    AutoConfig.register("openvla", OpenVLAConfig)\n    AutoImageProcessor.register(OpenVLAConfig, PrismaticImageProcessor)\n    AutoProcessor.register(OpenVLAConfig, PrismaticProcessor)\n    AutoModelForVision2Seq.register(OpenVLAConfig, OpenVLAForActionPrediction)\n\n    processor = AutoProcessor.from_pretrained(\n        "openvla/openvla-7b",\n        trust_remote_code=True,\n    )\n\n    quant = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_compute_dtype=torch.float16,\n        bnb_4bit_quant_type="nf4",\n    )\n\n    model = AutoModelForVision2Seq.from_pretrained(\n        "openvla/openvla-7b",\n        torch_dtype=torch.float16,\n        quantization_config=quant,\n        low_cpu_mem_usage=True,\n        trust_remote_code=True,\n    )\n\n    model = prepare_model_for_kbit_training(model)\n    model = get_peft_model(model, build_lora_config(condition))\n    audit_loaded_peft(model, condition)\n\n    for p in model.parameters():\n        if p.requires_grad and p.dtype != torch.float32:\n            p.data = p.data.float()\n        if p.device.type == "cpu":\n            p.data = p.data.to(device_id)\n\n    for b in model.buffers():\n        if b.device.type == "cpu":\n            b.data = b.data.to(device_id)\n\n    dtype_counts = {}\n    for p in model.parameters():\n        if p.requires_grad:\n            dtype_counts[str(p.dtype)] = dtype_counts.get(str(p.dtype), 0) + p.numel()\n\n    if set(dtype_counts) != {"torch.float32"}:\n        raise RuntimeError(f"Trainable adapter dtype mismatch: {dtype_counts}")\n\n    model = DDP(\n        model,\n        device_ids=[device_id],\n        find_unused_parameters=True,\n        gradient_as_bucket_view=True,\n    )\n    model._set_static_graph()\n\n    trainable_params = [p for p in model.parameters() if p.requires_grad]\n    optimizer = AdamW(trainable_params, lr=args.lr)\n\n    action_tokenizer = ActionTokenizer(processor.tokenizer)\n    batch_transform = RLDSBatchTransform(\n        action_tokenizer,\n        processor.tokenizer,\n        image_transform=processor.image_processor.apply_transform,\n        prompt_builder_fn=PurePromptBuilder,\n    )\n    dataset = RLDSDataset(\n        args.data_root,\n        "libero_object_no_noops",\n        batch_transform,\n        resize_resolution=tuple(model.module.config.image_sizes),\n        shuffle_buffer_size=10,\n        image_aug=True,\n    )\n\n    if distributed_state.is_main_process:\n        save_dataset_statistics(dataset.dataset_statistics, run_dir)\n\n    collator = PaddedCollatorForActionPrediction(\n        processor.tokenizer.model_max_length,\n        processor.tokenizer.pad_token_id,\n        padding_side="right",\n    )\n    loader = DataLoader(\n        dataset,\n        batch_size=args.batch_size,\n        sampler=None,\n        collate_fn=collator,\n        num_workers=0,\n    )\n\n    try:\n        scaler = torch.amp.GradScaler("cuda", enabled=True)\n    except Exception:\n        scaler = torch.cuda.amp.GradScaler(enabled=True)\n\n    metrics_path = run_dir / "training_metrics.csv"\n    if distributed_state.is_main_process:\n        with metrics_path.open("w", newline="") as f:\n            csv.writer(f).writerow([\n                "optimizer_step",\n                "loss",\n                "action_accuracy",\n                "action_l1",\n                "grad_norm_preclip",\n                "amp_scale_before",\n                "amp_scale_after",\n            ])\n\n    recent_losses = deque(maxlen=args.grad_accum)\n    recent_accs = deque(maxlen=args.grad_accum)\n    recent_l1s = deque(maxlen=args.grad_accum)\n\n    optimizer.zero_grad(set_to_none=True)\n    successful_steps = 0\n    micro_steps = 0\n    overflow_skips = 0\n    consecutive_overflows = 0\n\n    with tqdm.tqdm(total=args.steps, leave=True) as progress:\n        for batch in loader:\n            micro_steps += 1\n\n            with torch.autocast("cuda", dtype=torch.float16):\n                out: CausalLMOutputWithPast = model(\n                    input_ids=batch["input_ids"].to(device_id),\n                    attention_mask=batch["attention_mask"].to(device_id),\n                    pixel_values=batch["pixel_values"].to(\n                        device_id, dtype=torch.float16\n                    ),\n                    labels=batch["labels"],\n                )\n                loss = out.loss\n\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\n                    f"Non-finite loss at micro-step {micro_steps}: {float(loss)}"\n                )\n\n            scaler.scale(loss / args.grad_accum).backward()\n\n            action_logits = out.logits[\n                :,\n                model.module.vision_backbone.featurizer.patch_embed.num_patches:-1,\n            ]\n            action_preds = action_logits.argmax(dim=2)\n            action_gt = batch["labels"][:, 1:].to(action_preds.device)\n            mask = action_gt > action_tokenizer.action_token_begin_idx\n\n            if mask.sum().item() == 0:\n                raise RuntimeError("No action tokens found in current batch")\n\n            acc = (((action_preds == action_gt) & mask).sum().float() / mask.sum().float())\n\n            pred_cont = torch.tensor(\n                action_tokenizer.decode_token_ids_to_actions(\n                    action_preds[mask].detach().cpu().numpy()\n                )\n            )\n            gt_cont = torch.tensor(\n                action_tokenizer.decode_token_ids_to_actions(\n                    action_gt[mask].detach().cpu().numpy()\n                )\n            )\n            l1 = torch.nn.functional.l1_loss(pred_cont, gt_cont)\n\n            recent_losses.append(float(loss.item()))\n            recent_accs.append(float(acc.item()))\n            recent_l1s.append(float(l1.item()))\n\n            if micro_steps % args.grad_accum != 0:\n                continue\n\n            scaler.unscale_(optimizer)\n            scale_before = float(scaler.get_scale())\n\n            grad_norm = torch.nn.utils.clip_grad_norm_(\n                trainable_params,\n                max_norm=1.0,\n                error_if_nonfinite=False,\n            )\n\n            if not torch.isfinite(grad_norm):\n                overflow_skips += 1\n                consecutive_overflows += 1\n                new_scale = max(scale_before * 0.5, 1.0)\n                scaler.update(new_scale=new_scale)\n                optimizer.zero_grad(set_to_none=True)\n                print(\n                    f"[AMP OVERFLOW] step={successful_steps + 1} "\n                    f"scale={scale_before:.1f}->{new_scale:.1f}",\n                    flush=True,\n                )\n                if consecutive_overflows >= 8:\n                    raise FloatingPointError("8 consecutive AMP overflows")\n                continue\n\n            scaler.step(optimizer)\n            scaler.update()\n            scale_after = float(scaler.get_scale())\n            optimizer.zero_grad(set_to_none=True)\n\n            if scale_after < scale_before:\n                overflow_skips += 1\n                consecutive_overflows += 1\n                if consecutive_overflows >= 8:\n                    raise FloatingPointError("8 consecutive AMP-skipped updates")\n                continue\n\n            consecutive_overflows = 0\n            successful_steps += 1\n            progress.update(1)\n\n            smooth_loss = sum(recent_losses) / len(recent_losses)\n            smooth_acc = sum(recent_accs) / len(recent_accs)\n            smooth_l1 = sum(recent_l1s) / len(recent_l1s)\n\n            if distributed_state.is_main_process:\n                with metrics_path.open("a", newline="") as f:\n                    csv.writer(f).writerow([\n                        successful_steps,\n                        smooth_loss,\n                        smooth_acc,\n                        smooth_l1,\n                        float(grad_norm),\n                        scale_before,\n                        scale_after,\n                    ])\n\n            progress.set_description(\n                f"{condition} loss={smooth_loss:.3f} "\n                f"acc={smooth_acc:.3f} l1={smooth_l1:.3f}"\n            )\n\n            if successful_steps >= args.steps:\n                break\n\n    if successful_steps != args.steps:\n        raise RuntimeError(\n            f"{condition}: got {successful_steps} successful steps, expected {args.steps}"\n        )\n\n    if distributed_state.is_main_process:\n        processor.save_pretrained(run_dir)\n        adapter_dir.mkdir(parents=True, exist_ok=True)\n        model.module.save_pretrained(\n            adapter_dir,\n            safe_serialization=True,\n            save_embedding_layers=False,\n        )\n\n        completion = {\n            "condition": condition,\n            "successful_optimizer_steps": successful_steps,\n            "micro_steps": micro_steps,\n            "overflow_skips": overflow_skips,\n            "trainable_params": EXPECTED_TRAINABLE,\n            "mmd_selected_layers": MMD_SELECTED_LAYERS,\n            "final_adapter": str(adapter_dir),\n            "metrics_csv": str(metrics_path),\n            "dataset_statistics": str(run_dir / "dataset_statistics.json"),\n        }\n        (run_dir / "completion.json").write_text(\n            json.dumps(completion, indent=2)\n        )\n\n        print(\n            f"TRAINING SMOKE COMPLETE: {condition} "\n            f"{successful_steps}/{args.steps} steps",\n            flush=True,\n        )\n\n    dist.barrier()\n    dist.destroy_process_group()\n\n    del model\n    gc.collect()\n    torch.cuda.empty_cache()\n\n\nif __name__ == "__main__":\n    main()\n'
train_script_path = SCRATCH / "smoke_train.py"
train_script_path.write_text(TRAIN_SCRIPT)
ast.parse(TRAIN_SCRIPT)
import py_compile
py_compile.compile(str(train_script_path), doraise=True)

print("Training engine:", train_script_path)
print("Embedded script AST parse: PASS")
print("Written script compile: PASS")

Training engine: /kaggle/tmp/openvla_dual_smoke_v4/smoke_train.py
Embedded script AST parse: PASS
Written script compile: PASS


## 7. Run Uniform smoke

In [7]:
def run_condition(condition):
    cmd = [
        sys.executable, "-m", "torch.distributed.run",
        "--standalone", "--nnodes=1", "--nproc-per-node=1",
        str(train_script_path),
        "--condition", condition,
        "--data-root", str(DATA_DIR),
        "--output-root", str(RUNS_DIR),
        "--steps", str(SMOKE_STEPS),
        "--grad-accum", str(GRAD_ACCUM),
        "--batch-size", str(BATCH_SIZE),
        "--lr", "5e-4",
        "--seed", str(SEED),
    ]

    env = os.environ.copy()
    env.update({
        "CUDA_VISIBLE_DEVICES": "0",
        "HF_HOME": str(HF_HOME),
        "TRANSFORMERS_CACHE": str(HF_HOME / "transformers"),
        "TOKENIZERS_PARALLELISM": "false",
        "WANDB_MODE": "disabled",
    })

    free_gib = shutil.disk_usage("/kaggle/tmp").free / 1024**3
    print(f"Scratch free before {condition}: {free_gib:.1f} GiB")
    if free_gib < 4:
        raise RuntimeError(f"Insufficient scratch space before {condition}")

    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, env=env, cwd=OPENVLA_DIR)

run_condition("uniform")

Scratch free before uniform: 1116.1 GiB
$ /usr/bin/python3 -m torch.distributed.run --standalone --nnodes=1 --nproc-per-node=1 /kaggle/tmp/openvla_dual_smoke_v4/smoke_train.py --condition uniform --data-root /kaggle/tmp/openvla_dual_smoke_v4/datasets/openvla/modified_libero_rlds --output-root /kaggle/tmp/openvla_dual_smoke_v4/runs --steps 12 --grad-accum 4 --batch-size 1 --lr 5e-4 --seed 7


[W1001 16:02:25.826633417 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
[W1001 16:02:33.564377996 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Loading checkpoint shards: 100%|██████████| 3/3 [00:58<00:00, 19.55s/it]


ACTUAL PEFT STRUCTURE PASS: condition=uniform, trainable=110,828,288, layers=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]


/kaggle/tmp/openvla_dual_smoke_v4/smoke_train.py:215: UserWarning: You passed find_unused_parameters=true to DistributedDataParallel, `_set_static_graph` will detect unused parameters automatically, so you do not need to set find_unused_parameters=true, just be sure these unused parameters will not change during training loop while calling `_set_static_graph`.
  model._set_static_graph()


10/01 [16:05:09] INFO     | >> [*] Computing dataset           data_utils.py:232
                          statistics. This may take a bit, but                  
                          should only need to happen once.                      


100%|██████████| 454/454 [00:11<00:00, 40.87it/s]



######################################################################################
# Loading the following 1 datasets (incl. sampling weight):                         #
# libero_object_no_noops: ==================================================1.000000 #
######################################################################################

10/01 [16:05:21] INFO     | >> [*] Threads per Dataset: [1]       dataset.py:531
                 INFO     | >> [*] Reads per Dataset: [1]         dataset.py:532
                 INFO     | >> [*] Constructing datasets...       dataset.py:535
                 INFO     | >> [*] Applying frame transforms on   dataset.py:575
                          dataset...                                            
10/01 [16:05:23] INFO     | >> [*] Saved dataset statistics    data_utils.py:293
                          file at path                                          
                          /kaggle/tmp/openvla_dual_smoke_v4/ru                  
   

  0%|          | 0/12 [00:00<?, ?it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
W0000 00:00:1790870723.252155     721 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "CropAndResize" attr { key: "T" value { type: DT_FLOAT } } attr { key: "extrapolation_value" value { f: 0 } } attr { key: "method" value { s: "bilinear" } } inputs { dtype: DT_FLOAT shape { dim { size: 1 } dim { size: 224 } dim { size: 224 } dim { size: -7 } } } inputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: 4 } } } inputs { dtype: DT_INT32 shape { dim { size: -2 } } } inputs { dtype: DT_INT32 shape { dim { size: 2 } } } device { type: "CPU" vendor: "GenuineIntel" model: "101" frequency: 2000 num_cores: 4 environment { key: "cpu_instruction_set" value: "AVX SSE, SSE2, SSE3, SSSE3, SSE4.1, SSE4.2" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 32768 l2_cache_size: 1048576 l3_cache_size: 40370176 memory_size: 268435456 } outputs

TRAINING SMOKE COMPLETE: uniform 12/12 steps


/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


## 8. Run MMD-selective smoke

In [8]:
run_condition("mmd_selective")

Scratch free before mmd_selective: 1101.6 GiB
$ /usr/bin/python3 -m torch.distributed.run --standalone --nnodes=1 --nproc-per-node=1 /kaggle/tmp/openvla_dual_smoke_v4/smoke_train.py --condition mmd_selective --data-root /kaggle/tmp/openvla_dual_smoke_v4/datasets/openvla/modified_libero_rlds --output-root /kaggle/tmp/openvla_dual_smoke_v4/runs --steps 12 --grad-accum 4 --batch-size 1 --lr 5e-4 --seed 7


[W1001 16:06:34.075121766 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
[W1001 16:06:42.863486821 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Loading checkpoint shards: 100%|██████████| 3/3 [00:36<00:00, 12.01s/it]


ACTUAL PEFT STRUCTURE PASS: condition=mmd_selective, trainable=110,828,288, layers=[9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25]


/kaggle/tmp/openvla_dual_smoke_v4/smoke_train.py:215: UserWarning: You passed find_unused_parameters=true to DistributedDataParallel, `_set_static_graph` will detect unused parameters automatically, so you do not need to set find_unused_parameters=true, just be sure these unused parameters will not change during training loop while calling `_set_static_graph`.
  model._set_static_graph()


10/01 [16:07:30] INFO     | >> [*] Loading existing dataset    data_utils.py:208
                          statistics from                                       
                          /kaggle/tmp/openvla_dual_smoke_v4/da                  
                          tasets/openvla/modified_libero_rlds/                  
                          libero_object_no_noops/1.0.0/dataset                  
                          _statistics_c344e78839ccfa655e2747bf                  
                          1e72cf4279bc895fece27588d9e18f892c59                  
                          91a8.json.                                            

######################################################################################
# Loading the following 1 datasets (incl. sampling weight):                         #
# libero_object_no_noops: ==================================================1.000000 #
######################################################################################

   

...

mmd_selective loss=6.386 acc=0.214 l1=0.258:  25%|██▌       | 3/12 [00:21<01:07,  7.49s/it]

[AMP OVERFLOW] step=4 scale=32768.0->16384.0


mmd_selective loss=3.260 acc=0.393 l1=0.252: 100%|██████████| 12/12 [01:11<00:00,  5.94s/it]


TRAINING SMOKE COMPLETE: mmd_selective 12/12 steps


/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


## 9. Audit both outputs

In [9]:
import pandas as pd
import numpy as np

def audit_condition(condition):
    run_dir = RUNS_DIR / condition
    completion = json.loads((run_dir / "completion.json").read_text())
    metrics = pd.read_csv(run_dir / "training_metrics.csv")

    assert completion["condition"] == condition
    assert completion["successful_optimizer_steps"] == SMOKE_STEPS
    assert completion["trainable_params"] == EXPECTED_TRAINABLE_PARAMS
    assert len(metrics) == SMOKE_STEPS
    assert int(metrics["optimizer_step"].iloc[-1]) == SMOKE_STEPS

    numeric = [
        "loss", "action_accuracy", "action_l1",
        "grad_norm_preclip", "amp_scale_before", "amp_scale_after"
    ]
    assert np.isfinite(metrics[numeric].to_numpy()).all()
    assert metrics["action_accuracy"].between(0, 1).all()

    adapter = Path(completion["final_adapter"])
    stats = Path(completion["dataset_statistics"])
    assert (adapter / "adapter_config.json").exists()
    assert (
        (adapter / "adapter_model.safetensors").exists()
        or (adapter / "adapter_model.bin").exists()
    )
    assert stats.exists()

    return {
        "condition": condition,
        "completion": completion,
        "first_loss": float(metrics["loss"].iloc[0]),
        "last_loss": float(metrics["loss"].iloc[-1]),
        "first_accuracy": float(metrics["action_accuracy"].iloc[0]),
        "last_accuracy": float(metrics["action_accuracy"].iloc[-1]),
    }

uniform_result = audit_condition("uniform")
mmd_result = audit_condition("mmd_selective")

print(json.dumps({
    "uniform": uniform_result,
    "mmd_selective": mmd_result,
}, indent=2))
print("TRAINING OUTPUT AUDIT: PASS")

{
  "uniform": {
    "condition": "uniform",
    "completion": {
      "condition": "uniform",
      "successful_optimizer_steps": 12,
      "micro_steps": 48,
      "overflow_skips": 0,
      "trainable_params": 110828288,
      "mmd_selected_layers": [
        9,
        10,
        11,
        12,
        13,
        14,
        15,
        16,
        17,
        18,
        19,
        20,
        21,
        22,
        23,
        25
      ],
      "final_adapter": "/kaggle/tmp/openvla_dual_smoke_v4/runs/uniform/final_adapter",
      "metrics_csv": "/kaggle/tmp/openvla_dual_smoke_v4/runs/uniform/training_metrics.csv",
      "dataset_statistics": "/kaggle/tmp/openvla_dual_smoke_v4/runs/uniform/dataset_statistics.json"
    },
    "first_loss": 9.890597105026243,
    "last_loss": 3.3646469712257385,
    "first_accuracy": 0.1785714365541935,
    "last_accuracy": 0.2500000074505806
  },
  "mmd_selective": {
    "condition": "mmd_selective",
    "completion": {
      "condition": "mmd

## 10. Fresh-process adapter reload + action inference

In [10]:
RELOAD_SCRIPT = '\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom peft import PeftModel\nfrom transformers import (\n    AutoModelForVision2Seq,\n    AutoProcessor,\n    BitsAndBytesConfig,\n)\n\nap = argparse.ArgumentParser()\nap.add_argument("--condition", required=True)\nap.add_argument("--adapter", type=Path, required=True)\nap.add_argument("--stats", type=Path, required=True)\nargs = ap.parse_args()\n\nquant = BitsAndBytesConfig(\n    load_in_4bit=True,\n    bnb_4bit_compute_dtype=torch.float16,\n    bnb_4bit_quant_type="nf4",\n)\n\nprocessor = AutoProcessor.from_pretrained(\n    "openvla/openvla-7b",\n    trust_remote_code=True,\n)\nbase = AutoModelForVision2Seq.from_pretrained(\n    "openvla/openvla-7b",\n    trust_remote_code=True,\n    torch_dtype=torch.float16,\n    quantization_config=quant,\n    low_cpu_mem_usage=True,\n)\nmodel = PeftModel.from_pretrained(base, args.adapter, is_trainable=False)\nmodel.eval()\n\nstats = json.loads(args.stats.read_text())\nif "libero_object_no_noops" in stats:\n    key = "libero_object_no_noops"\nelif "libero_object" in stats:\n    key = "libero_object"\nelse:\n    raise KeyError(f"Unexpected stats keys: {list(stats)}")\n\ncore = model.base_model.model\ncore.norm_stats = stats\n\nimg = np.zeros((224, 224, 3), dtype=np.uint8)\nimg[:, :112, 0] = 255\nimg[:, 112:, 2] = 255\n\nprompt = (\n    "In: What action should the robot take to "\n    "pick up the alphabet soup and place it in the basket?\\nOut:"\n)\ninputs = processor(prompt, Image.fromarray(img).convert("RGB"))\nmoved = {}\nfor k, v in inputs.items():\n    if torch.is_tensor(v):\n        moved[k] = (\n            v.to("cuda:0", dtype=torch.float16)\n            if torch.is_floating_point(v)\n            else v.to("cuda:0")\n        )\n    else:\n        moved[k] = v\n\nwith torch.inference_mode():\n    action = core.predict_action(\n        **moved,\n        unnorm_key=key,\n        do_sample=False,\n    )\n\naction = np.asarray(action, dtype=np.float64)\nassert action.shape == (7,), action.shape\nassert np.all(np.isfinite(action)), action\n\nprint(f"FRESH RELOAD PASS: {args.condition}")\nprint("Action:", action.tolist())\n'
reload_script_path = SCRATCH / "verify_reload.py"
reload_script_path.write_text(RELOAD_SCRIPT)
ast.parse(RELOAD_SCRIPT)
import py_compile
py_compile.compile(str(reload_script_path), doraise=True)

def verify_reload(condition):
    completion = json.loads(
        (RUNS_DIR / condition / "completion.json").read_text()
    )
    cmd = [
        sys.executable, "-u", str(reload_script_path),
        "--condition", condition,
        "--adapter", completion["final_adapter"],
        "--stats", completion["dataset_statistics"],
    ]
    env = os.environ.copy()
    env.update({
        "CUDA_VISIBLE_DEVICES": "0",
        "HF_HOME": str(HF_HOME),
        "TRANSFORMERS_CACHE": str(HF_HOME / "transformers"),
        "TOKENIZERS_PARALLELISM": "false",
    })
    subprocess.run(cmd, check=True, env=env)

verify_reload("uniform")
verify_reload("mmd_selective")

print("FRESH-PROCESS RELOAD/INFERENCE: PASS")

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
A new version of the following files was downloaded from https://huggingface.co/openvla/openvla-7b:
- processing_prismatic.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/openvla/openvla-7b:
- configuration_prismatic.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new ver

FRESH RELOAD PASS: uniform
Action: [0.0942804712875216, -0.002584043671103098, -0.1470588235294118, -3.151186541014428e-05, 0.045247897608023085, -0.02000000302990279, 0.996078431372549]


/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
A new version of the following files was downloaded from https://huggingface.co/openvla/openvla-7b:
- processing_prismatic.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/openvla/openvla-7b:
- configuration_prismatic.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new ver

FRESH RELOAD PASS: mmd_selective
Action: [0.16487921008876727, -0.30605043383205643, 0.014705882352941124, -0.010102940482251785, 0.008777310114865211, -0.009369750919879627, 0.0]
FRESH-PROCESS RELOAD/INFERENCE: PASS


## 11. Persist compact final report

In [11]:
report = {
    "status": "PASS",
    "purpose": "pre-institutional dual-condition smoke",
    "openvla_commit": OPENVLA_COMMIT,
    "dataset": "libero_object_no_noops",
    "mmd_selected_layers": MMD_SELECTED_LAYERS,
    "trainable_params_each": EXPECTED_TRAINABLE_PARAMS,
    "smoke_steps_each": SMOKE_STEPS,
    "effective_batch": BATCH_SIZE * GRAD_ACCUM,
    "uniform": uniform_result,
    "mmd_selective": mmd_result,
    "validated": [
        "Uniform exact 32-layer LoRA structure",
        "MMD exact robust 16-layer LoRA structure",
        "equal trainable-parameter budget",
        "12 successful optimiser steps per condition",
        "FP16 GradScaler",
        "gradient accumulation",
        "gradient clipping",
        "adapter-only save",
        "fresh-process adapter reload",
        "finite 7-D action inference after reload",
    ],
}

report_path = PERSIST / "FINAL_SMOKE_REPORT.json"
report_path.write_text(json.dumps(report, indent=2))

print("=" * 78)
print("KAGGLE DUAL-CONDITION SMOKE V4: PASS")
print("=" * 78)
print("Report:", report_path)
print(f"Scratch free: {shutil.disk_usage('/kaggle/tmp').free / 1024**3:.1f} GiB")
print(f"Working free: {shutil.disk_usage('/kaggle/working').free / 1024**3:.1f} GiB")

KAGGLE DUAL-CONDITION SMOKE V4: PASS
Report: /kaggle/working/openvla_dual_smoke_v4_results/FINAL_SMOKE_REPORT.json
Scratch free: 1101.2 GiB
Working free: 19.5 GiB


## What to send back

Send the executed notebook or:

`/kaggle/working/openvla_dual_smoke_v4_results/FINAL_SMOKE_REPORT.json`

A PASS means the experiment engine worked end-to-end for both conditions. It does not
mean either 12-step policy is useful yet.